# (강의) Model, Dataset, Trainer

신록예찬  
2024-11-08

`-` 예측을 하는 방법 3가지

-   `classifier`로 예측하기
-   `trainer`로 예측하기
-   `model`로 예측하기

`-` 자료가 저장 혹은 정리된 형태는 거의 무한대의 경우의 수가 있음.
그렇지만 transformers를 이용하여 분석할 경우는 크게 아래의 3가지로
나누어짐.

-   원시형태의 자료: `classifier`의 입력으로 사용되는 **데이터**
-   전처리된 자료: `trainer`의 입력으로 사용되는 **데이터**
-   더 전처리된 자료: `model`의 입력으로 사용되는 **데이터**

# 1. 강의영상

# 2. Imports

In [99]:
import transformers
import torch
import datasets
import mp2024pkg
from rich import print as rprint

# 3. Model 입력

`-` 아래중 하나의 방법순으로..

-   방법1: `model.forward?` 에서 시그니처를 확인
-   방법2: `model.forward?` 에서 사용예제를 확인
-   방법3: 인터넷을 활용한 외부 자료 확인 (공식문서, 공식튜토리얼,
    신뢰할만한 블로그, ChatGPT등)
-   방법4: `model.forward??` 를 보고 모든 코드를 뜯어봄 \<— 하지마세요

`-` 모델의 입력이 어떤형태로 정리되어야 하는지 알아내는 확실한 방법은
없음

-   방법1,2,3 은 다른사람의 호의에 기대해야함.
-   방법4는 사실상 불가능

`# 예제1` – 텍스트분류

In [12]:
model1 = transformers.AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

In [13]:
model1.config

DistilBertConfig {
  "_name_or_path": "distilbert/distilbert-base-uncased",
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.45.2",
  "vocab_size": 30522
}

In [ ]:
mp2024pkg.signature(model1.forward)

In [14]:
#model1.forward?

    </Tip>

    Args:
        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
            Indices of input sequence tokens in the vocabulary.

            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and
            [`PreTrainedTokenizer.__call__`] for details.

            [What are input IDs?](../glossary#input-ids)
        attention_mask (`torch.FloatTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:

            - 1 for tokens that are **not masked**,
            - 0 for tokens that are **masked**.

            [What are attention masks?](../glossary#attention-mask)
        head_mask (`torch.FloatTensor` of shape `(num_heads,)` or `(num_layers, num_heads)`, *optional*):
            Mask to nullify selected heads of the self-attention modules. Mask values selected in `[0, 1]`:

            - 1 indicates the head is **not masked**,
            - 0 indicates the head is **masked**.

        inputs_embeds (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`, *optional*):
            Optionally, instead of passing `input_ids` you can choose to directly pass an embedded representation. This
            is useful if you want more control over how to convert `input_ids` indices into associated vectors than the
            model's internal embedding lookup matrix.
        output_attentions (`bool`, *optional*):
            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned
            tensors for more detail.
        output_hidden_states (`bool`, *optional*):
            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for
            more detail.
        return_dict (`bool`, *optional*):
            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.

        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).

In [30]:
model1.forward(
    input_ids = torch.tensor([[1,2,3],[2,3,4]]),
    labels = torch.tensor([1,0])
)

SequenceClassifierOutput(loss=tensor(0.6917, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0143,  0.0154],
        [-0.0138,  0.0097]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [36]:
model1_input = dict(
    input_ids = torch.tensor([[1,2,3],[2,3,4]]),
    labels = torch.tensor([1,0])
)
model1.forward(**model1_input)

SequenceClassifierOutput(loss=tensor(0.6917, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0143,  0.0154],
        [-0.0138,  0.0097]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [37]:
model1_input = dict(
    input_ids = torch.tensor([[1,2,3],[2,3,4]]),
    labels = torch.tensor([1,0])
)
model1(**model1_input)

SequenceClassifierOutput(loss=tensor(0.6917, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0143,  0.0154],
        [-0.0138,  0.0097]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

`#`

`# 예제2` – 이미지분류

In [19]:
model2 = transformers.AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=3
)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

In [20]:
model2.config

ViTConfig {
  "_name_or_path": "google/vit-base-patch16-224-in21k",
  "architectures": [
    "ViTModel"
  ],
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "image_size": 224,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": true,
  "transformers_version": "4.45.2"
}

In [78]:
mp2024pkg.signature(model2.forward)

In [22]:
#model2.forward?

    </Tip>

    Args:
        pixel_values (`torch.FloatTensor` of shape `(batch_size, num_channels, height, width)`):
            Pixel values. Pixel values can be obtained using [`AutoImageProcessor`]. See [`ViTImageProcessor.__call__`]
            for details.

        head_mask (`torch.FloatTensor` of shape `(num_heads,)` or `(num_layers, num_heads)`, *optional*):
            Mask to nullify selected heads of the self-attention modules. Mask values selected in `[0, 1]`:

            - 1 indicates the head is **not masked**,
            - 0 indicates the head is **masked**.

        output_attentions (`bool`, *optional*):
            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned
            tensors for more detail.
        output_hidden_states (`bool`, *optional*):
            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for
            more detail.
        interpolate_pos_encoding (`bool`, *optional*):
            Whether to interpolate the pre-trained position encodings.
        return_dict (`bool`, *optional*):
            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.

        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the image classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).

In [80]:
model2(
    pixel_values = torch.randn(2,3,224,224),
    labels = torch.tensor([0,0])
)

ImageClassifierOutput(loss=tensor(1.0836, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0836, -0.1265, -0.1150],
        [-0.0979, -0.1208, -0.0916]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [81]:
model2_input = dict(
    pixel_values = torch.randn(2,3,224,224),
    labels = torch.tensor([0,0])
)
model2(**model2_input)

ImageClassifierOutput(loss=tensor(1.1028, grad_fn=<NllLossBackward0>), logits=tensor([[-0.1178, -0.1516, -0.0915],
        [-0.1061, -0.1385, -0.0442]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

`#`

`# 예제3` – 동영상분류

In [170]:
model3 = transformers.VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base",
)

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

In [171]:
model3.config

VideoMAEConfig {
  "_name_or_path": "MCG-NJU/videomae-base",
  "architectures": [
    "VideoMAEForPreTraining"
  ],
  "attention_probs_dropout_prob": 0.0,
  "decoder_hidden_size": 384,
  "decoder_intermediate_size": 1536,
  "decoder_num_attention_heads": 6,
  "decoder_num_hidden_layers": 4,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 224,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "videomae",
  "norm_pix_loss": true,
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_frames": 16,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": true,
  "torch_dtype": "float32",
  "transformers_version": "4.45.2",
  "tubelet_size": 2,
  "use_mean_pooling": false
}

In [172]:
mp2024pkg.signature(model3.forward)

In [173]:
#model3.forward?

    </Tip>

    Args:
        pixel_values (`torch.FloatTensor` of shape `(batch_size, num_frames, num_channels, height, width)`):
            Pixel values. Pixel values can be obtained using [`AutoImageProcessor`]. See
            [`VideoMAEImageProcessor.__call__`] for details.

        head_mask (`torch.FloatTensor` of shape `(num_heads,)` or `(num_layers, num_heads)`, *optional*):
            Mask to nullify selected heads of the self-attention modules. Mask values selected in `[0, 1]`:

            - 1 indicates the head is **not masked**,
            - 0 indicates the head is **masked**.

        output_attentions (`bool`, *optional*):
            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned
            tensors for more detail.
        output_hidden_states (`bool`, *optional*):
            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for
            more detail.
        return_dict (`bool`, *optional*):
            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.

        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the image classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).```

    ::: {#cell-34 .cell execution_count=174}
    ``` {.python .cell-code}
    model3(
        pixel_values = torch.randn(2,16,3,224,224),
        labels = torch.tensor([1,0])
    )

    ImageClassifierOutput(loss=tensor(0.6963, grad_fn=<NllLossBackward0>), logits=tensor([[0.1127, 0.0495],
            [0.1182, 0.0661]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

:::

In [175]:
model3_input = dict(
    pixel_values = torch.randn(2,16,3,224,224),
    labels = torch.tensor([1,0])
)
model3(**model3_input)

ImageClassifierOutput(loss=tensor(0.7447, grad_fn=<NllLossBackward0>), logits=tensor([[0.2101, 0.0184],
        [0.1063, 0.1115]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

`#`

# 4. Model 사용 연습

## A. 텍스트

In [167]:
tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
    0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [168]:
model1

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [193]:
mp2024pkg.signature(model1.forward)

`# 예제1`

In [ ]:
imdb = datasets.load_dataset('imdb')
d = imdb['train'].select(range(3))
d

`(풀이)`

In [157]:
model1(
    input_ids = torch.tensor(tokenizer(d['text'])['input_ids']),
    labels = torch.tensor([0,0,1])
    
)

In [158]:
mp2024pkg.show_list(tokenizer(d['text'])['input_ids'])

Level 1 - Type: list, Length: 3, Content: [[101, 1045, 12524, 1045, ... , 7987, 1013, 1028, 102]]
     Level 2 - Type: list, Length: 363, Content: [101, 1045, 12524, 1045,  ... 7, 1037, 5436, 1012, 102]
     Level 2 - Type: list, Length: 304, Content: [101, 1000, 1045, 2572, 8 ... 5, 1055, 4230, 1012, 102]
     Level 2 - Type: list, Length: 133, Content: [101, 2065, 2069, 2000, 4 ... 6, 7987, 1013, 1028, 102]

In [159]:
mp2024pkg.show_list(tokenizer(d['text'],padding=True)['input_ids'])

Level 1 - Type: list, Length: 3, Content: [[101, 1045, 12524, 1045, ...  0, 0, 0, 0, 0, 0, 0, 0]]
     Level 2 - Type: list, Length: 363, Content: [101, 1045, 12524, 1045,  ... 7, 1037, 5436, 1012, 102]
     Level 2 - Type: list, Length: 363, Content: [101, 1000, 1045, 2572, 8 ... , 0, 0, 0, 0, 0, 0, 0, 0]
     Level 2 - Type: list, Length: 363, Content: [101, 2065, 2069, 2000, 4 ... , 0, 0, 0, 0, 0, 0, 0, 0]

In [176]:
model1(
    input_ids = torch.tensor(tokenizer(d['text'],padding=True)['input_ids']),
    labels = torch.tensor([0,0,1])
)

SequenceClassifierOutput(loss=tensor(0.6759, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0548, -0.1004],
        [-0.0423, -0.0394],
        [-0.0736, -0.0115]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

`#`

`# 예제2`

In [191]:
emotion = datasets.load_dataset('emotion')

In [192]:
d = emotion['train'].select(range(3))
d

Dataset({
    features: ['text', 'label'],
    num_rows: 3
})

`(풀이)`

In [190]:
model1(
    torch.tensor(tokenizer(d['text'],padding=True)['input_ids'])
)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.0089, -0.0803],
        [-0.0257, -0.0503],
        [-0.0180, -0.0846]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

> 진짜 최소로 타이핑하려면 이렇게만 해도 된다.

`#`

## B. 이미지

## C. 동영상

# 4. Dataset

## A. 형식이해

`# 예제1`

In [114]:
imdb = datasets.load_dataset('imdb')
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [116]:
d = imdb['train'].select(range(3))
d

Dataset({
    features: ['text', 'label'],
    num_rows: 3
})

In [117]:
이해방식1 = [d[0], d[1], d[2]]
rprint(이해방식1)
# = [Dict_1, Dict_2, Dict_3] 
# = [{'text': xxx, 'label':yyy},
#    {'text': xxxx, 'label':yyyy},
#    {'text': xxxxx, 'label':yyyyy}]

[ 
 { 
 'text' : 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it 
 when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to 
 enter this country, therefore being a fan of films considered "controversial" I really had to see this for 
 myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn 
 everything she can about life. In particular she wants to focus her attentions to making some sort of documentary 
 on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the 
 United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, 
 she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW 
 is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, 
 even then it\'s not shot like some cheaply made porno. While my countrymen mind find it shocking, in reality sex 
 and nudity are a major staple in Swedish cinema. Even Ingmar Bergman, arguably their answer to good old boy John 
 Ford, had sex scenes in his films.<br /><br />I do commend the filmmakers for the fact that any sex shown in the 
 film is shown for artistic purposes rather than just to shock people and make money to be shown in pornographic 
 theaters in America. I AM CURIOUS-YELLOW is a good film for anyone wanting to study the meat and potatoes (no pun 
 intended) of Swedish cinema. But really, this film doesn\'t have much of a plot.' , 
 'label' : 0 
 } , 
 { 
 'text' : '"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn\'t matter what one\'s 
 political views are because this film can hardly be taken seriously on any level. As for the claim that frontal 
 male nudity is an automatic NC-17, that isn\'t true. I\'ve seen R-rated films with male nudity. Granted, they only 
 offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because 
 they don\'t exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris 
 in sight. And those pretentious indie movies like The Brown Bunny, in which we\'re treated to the site of Vincent 
 Gallo\'s throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) 
 "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious 
 anatomical difference between men and women: there are no genitals on display when actresses appears nude, and the 
 same cannot be said for a man. In fact, you generally won\'t see female genitals in an American film in anything 
 short of porn or explicit erotica. This alleged double-standard is less a double standard than an admittedly 
 depressing ability to come to terms culturally with the insides of women\'s bodies.' , 
 'label' : 0 
 } , 
 { 
 'text' : "If only to avoid making this type of film in the future. This film is interesting as an experiment 
 but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many 
 IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives 
 (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br 
 /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />" ,
 'label' : 0 
 } 
 ]

In [118]:
이해방식2 = {'text':d['text'], 'label':d['label']}
rprint(이해방식2)

{ 
 'text' : [ 
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it 
 was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this 
 country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br 
 />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about 
 life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede 
 thought about certain political issues such as the Vietnam War and race issues in the United States. In between 
 asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama 
 teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this 
 was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it\'s not shot 
 like some cheaply made porno. While my countrymen mind find it shocking, in reality sex and nudity are a major 
 staple in Swedish cinema. Even Ingmar Bergman, arguably their answer to good old boy John Ford, had sex scenes in 
 his films.<br /><br />I do commend the filmmakers for the fact that any sex shown in the film is shown for artistic 
 purposes rather than just to shock people and make money to be shown in pornographic theaters in America. I AM 
 CURIOUS-YELLOW is a good film for anyone wanting to study the meat and potatoes (no pun intended) of Swedish 
 cinema. But really, this film doesn\'t have much of a plot.' , 
 '"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn\'t matter what one\'s 
 political views are because this film can hardly be taken seriously on any level. As for the claim that frontal 
 male nudity is an automatic NC-17, that isn\'t true. I\'ve seen R-rated films with male nudity. Granted, they only 
 offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because 
 they don\'t exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris 
 in sight. And those pretentious indie movies like The Brown Bunny, in which we\'re treated to the site of Vincent 
 Gallo\'s throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) 
 "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious 
 anatomical difference between men and women: there are no genitals on display when actresses appears nude, and the 
 same cannot be said for a man. In fact, you generally won\'t see female genitals in an American film in anything 
 short of porn or explicit erotica. This alleged double-standard is less a double standard than an admittedly 
 depressing ability to come to terms culturally with the insides of women\'s bodies.' , 
 "If only to avoid making this type of film in the future. This film is interesting as an experiment but 
 tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many 
 IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives 
 (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br 
 /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />" 
 ] ,
 'label' : [ 0 , 0 , 0 ] 
 }

`#`

`#`

In [121]:
mbti = datasets.Dataset.from_csv("./mbti_1.csv")
mbti

Dataset({
    features: ['type', 'posts'],
    num_rows: 8675
})

In [122]:
mbti

Dataset({
    features: ['type', 'posts'],
    num_rows: 8675
})